# Improved Website Image Extraction: IKEA US

This notebook improves the original workflow by fetching the IKEA US homepage, extracting image URLs, downloading up to 10 images, and creating a ZIP archive.

## Improvements
- Reuses a `requests.Session` and retries temporary HTTP failures.
- Handles additional lazy-loading attributes and parses `srcset` more carefully.
- Resolves relative URLs and restricts downloads to IKEA-owned domains.
- Checks HTTP status and image content type before saving.
- Uses `pathlib` for cleaner, cross-platform file handling.
- Works in Google Colab or a local Jupyter environment.

**Note:** Some website images may be loaded dynamically with JavaScript and will not appear in the initial HTML. Always respect the website's terms of use, robots instructions, and reasonable request rates.

In [ ]:
# If needed, install the required packages
%pip -q install requests beautifulsoup4

In [ ]:
from pathlib import Path
from urllib.parse import urljoin, urlparse
import mimetypes
import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

BASE_URL = "https://www.ikea.com/us/en/"
OUTPUT_DIR = Path("ikea_images")
MAX_IMAGES = 10
TIMEOUT = 30

# Reuse one session and retry selected transient errors.
session = requests.Session()
retry_strategy = Retry(
    total=3,
    connect=3,
    read=3,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"]
)
adapter = HTTPAdapter(max_retries=retry_strategy)
session.mount("https://", adapter)
session.mount("http://", adapter)
session.headers.update({
    "User-Agent": "Mozilla/5.0 (compatible; educational image extraction notebook)"
})

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output folder: {OUTPUT_DIR.resolve()}")

## Step 1 — Fetch the webpage and extract image URLs

The function below checks common image attributes used by standard and lazy-loaded images. For `srcset`, it selects the candidate with the largest declared width when width descriptors are available.

In [ ]:
def choose_srcset_candidate(srcset):
    """Return the largest-width candidate from a srcset string."""
    if not srcset:
        return None

    candidates = []
    for item in srcset.split(","):
        parts = item.strip().split()
        if not parts:
            continue
        image_url = parts[0]
        width = 0
        if len(parts) > 1 and parts[1].endswith("w"):
            try:
                width = int(parts[1][:-1])
            except ValueError:
                width = 0
        candidates.append((width, image_url))

    if not candidates:
        return None
    return max(candidates, key=lambda item: item[0])[1]


response = session.get(BASE_URL, timeout=TIMEOUT)
response.raise_for_status()
print("HTTP status:", response.status_code)

soup = BeautifulSoup(response.text, "html.parser")
image_urls = []
seen_urls = set()

image_attributes = [
    "src", "data-src", "data-lazy-src", "data-original",
    "data-image", "data-srcset"
]

for img in soup.find_all("img"):
    raw_url = None

    for attribute in image_attributes:
        value = img.get(attribute)
        if value:
            # srcset-like values should be parsed as candidate lists.
            if "srcset" in attribute:
                raw_url = choose_srcset_candidate(value)
            else:
                raw_url = value.strip()
            if raw_url:
                break

    if not raw_url:
        raw_url = choose_srcset_candidate(img.get("srcset"))

    if not raw_url or raw_url.startswith(("data:", "blob:")):
        continue

    absolute_url = urljoin(BASE_URL, raw_url)
    parsed = urlparse(absolute_url)
    hostname = (parsed.hostname or "").lower()

    # Keep only IKEA domains; this avoids downloading unrelated third-party images.
    if parsed.scheme not in ("http", "https"):
        continue
    if hostname == "ikea.com" or hostname.endswith(".ikea.com"):
        if absolute_url not in seen_urls:
            seen_urls.add(absolute_url)
            image_urls.append(absolute_url)

print(f"Unique IKEA image URLs found: {len(image_urls)}")
for image_url in image_urls[:MAX_IMAGES]:
    print(image_url)

## Step 2 — Download up to 10 images

Only responses identified as common image formats are saved. Failed downloads are reported without stopping the remaining downloads.

In [ ]:
MIME_TO_EXTENSION = {
    "image/jpeg": ".jpg",
    "image/png": ".png",
    "image/webp": ".webp",
    "image/gif": ".gif",
    "image/avif": ".avif"
}

downloaded_files = []
failed_downloads = []

for index, image_url in enumerate(image_urls[:MAX_IMAGES], start=1):
    try:
        with session.get(
            image_url,
            stream=True,
            timeout=TIMEOUT
        ) as image_response:
            image_response.raise_for_status()
            content_type = image_response.headers.get("Content-Type", "")
            mime_type = content_type.split(";")[0].strip().lower()

            extension = MIME_TO_EXTENSION.get(mime_type)
            if extension is None:
                failed_downloads.append((image_url, f"Unsupported content type: {mime_type or 'unknown'}"))
                continue

            file_path = OUTPUT_DIR / f"ikea_{index:02d}{extension}"
            with file_path.open("wb") as image_file:
                for chunk in image_response.iter_content(chunk_size=64 * 1024):
                    if chunk:
                        image_file.write(chunk)

            # Remove empty files if a server returned an unexpected empty response.
            if file_path.stat().st_size == 0:
                file_path.unlink(missing_ok=True)
                failed_downloads.append((image_url, "Downloaded file was empty"))
                continue

            downloaded_files.append(file_path)
            print(f"Downloaded: {file_path.name} ({file_path.stat().st_size:,} bytes)")

    except requests.RequestException as error:
        failed_downloads.append((image_url, str(error)))
        print(f"Download failed: {image_url}\n  Reason: {error}")

print("\nDownload summary")
print("Successfully downloaded:", len(downloaded_files))
print("Failed or skipped:", len(failed_downloads))

## Step 3 — Create a ZIP file

The ZIP archive is saved beside the image folder. In Google Colab, the notebook will offer the archive as a download. In local Jupyter, use the printed path to access it.

In [ ]:
import shutil

zip_base = OUTPUT_DIR.parent / OUTPUT_DIR.name
zip_path = Path(shutil.make_archive(str(zip_base), "zip", root_dir=OUTPUT_DIR))
print(f"ZIP archive created: {zip_path.resolve()}")

# Google Colab download; this import is skipped in ordinary local Jupyter.
try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print("Not running in Google Colab. Open the ZIP file from the path above.")

## Step 4 — Review the results

This final cell lists the downloaded files and any failed or skipped URLs so that the results can be checked.

In [ ]:
print("Downloaded image files:")
for file_path in downloaded_files:
    print(f"- {file_path.name}")

if failed_downloads:
    print("\nFailed or skipped URLs:")
    for failed_url, reason in failed_downloads:
        print(f"- {failed_url}\n  {reason}")
else:
    print("\nNo failed or skipped downloads were recorded.")